In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import scipy.stats as stats
import seaborn as sns

### Module 2.1: The Core Limitation of Python's Lists

Before we can truly appreciate the **power and necessity of `NumPy`** in scientific computing, it's crucial to understand the **fundamental design choices** of a standard Python `list`—and why these choices, while offering incredible flexibility, become significant *bottlenecks* for numerical operations.

## 1. The Nature of Python Lists

A Python `list` is often lauded for its **versatility**.  
You can effortlessly create a list containing a mix of different data types:

```python
my_list = [1, "hello", 99.5, [10, 20]]
````

This apparent simplicity hides a **complex internal structure**.
A Python list is **not** a contiguous block of raw data values.
Instead, think of it as a **collection of pointers**—each element is a **reference** (a memory address) to a distinct, self-contained Python object stored somewhere in memory.

These objects can be integers, strings, floats, or even other lists, scattered across different memory locations.

## 2. Under the Hood: What Really Happens

When you access `my_list[0]`, Python doesn’t directly retrieve the value `1`.
Instead, it performs several steps:

1. **Lookup:** Finds the memory address stored at the first position in `my_list`.
2. **Dereference:** Follows that pointer to locate the actual Python object.
3. **Interpretation:** Retrieves a complete **`PyObject` structure**, which includes:

   * the **value** itself (`1`);
   * its **type information** (`int`);
   * a **reference count** for memory management;
   * and other metadata.

Only after this chain of lookups does Python extract the raw numerical value.

## 3. Why Lists Fall Short for Numerical Computing

The flexibility of Python lists directly **conflicts** with the demands of high-performance numerical mathematics.
Numerical computing thrives on **homogeneity** and **tightly packed data**.

Imagine a CPU performing operations on a sequence of numbers.
Ideally, it wants to *stream through memory*, processing numbers one after another without interruption.
When data is homogeneous (e.g., all `float64`) and stored contiguously, the CPU can exploit two major optimisations:

### 3.1 Vectorised CPU Instructions (SIMD)

Modern CPUs can perform **Single Instruction, Multiple Data (SIMD)** operations—applying the same instruction to multiple values at once.
This offers *massive speedups*, but only when the data is stored **predictably and contiguously** in memory.

### 3.2 Cache Locality

When data is **contiguous**, accessing one element brings nearby elements into the CPU’s **cache memory**.
Subsequent accesses are much faster because they’re served from cache rather than from main memory.

With a Python list, the interpreter is forced into a process called *dynamic dispatch* for every single operation. For each element, it must:
1.  Retrieve the pointer.
2.  Follow the pointer to the `PyObject`.
3.  Inspect the `PyObject` to determine its type.
4.  Find the correct, type-specific implementation of the operation (e.g., integer multiplication vs. float multiplication).
5.  Execute the operation.

If you're working with millions of numbers, these tiny, repeated "bookkeeping" checks and memory dereferences accumulate, dominating the total execution time.

Let's consider a common mathematical operation: doubling a list of numbers.
Suppose you have a list representing financial revenues:
`revenues = [100, 200, 300]`

From a mathematical perspective (as a Statistician), you intuitively want to perform scalar multiplication:
$2 \times [100, 200, 300] = [200, 400, 600]$

However, the standard Python interpreter, designed for general-purpose operations, interprets the `*` operator differently for lists:
`revenues * 2`
`[100, 200, 300, 100, 200, 300]`

This is **list concatenation** (repeating the list), not element-wise **multiplication**. Python's built-in list type simply doesn't "understand" mathematical vector operations. To achieve the desired element-wise multiplication, you are forced to explicitly iterate, typically using a `for` loop or a list comprehension:
`[x * 2 for x in revenues]`

While this produces the correct result, it is inherently slow for several critical reasons:
1.  **Interpreter Overhead (The "Python Loop" Problem):** As discussed, the Python interpreter must manage the loop, retrieve each `PyObject`, check its type, and dispatch the appropriate multiplication method for *every single element*. This constant context switching and type checking is a significant performance drain, even for seemingly simple operations.
2.  **Poor Memory Layout (Scattered Data):** The numbers in `revenues` are not stored contiguously. They are individual `PyObject`s potentially scattered throughout memory. This forces the CPU to "chase pointers" all over the place, leading to frequent cache misses and preventing it from utilizing its high-speed vectorized processing capabilities.
3.  **Missing "Contracts" (Lack of Guarantees):** A Python list offers no guarantee about the types of its contents. It could contain numbers, strings, or a mix. This means the interpreter cannot make any assumptions that would allow for optimization. It must always be prepared for the most general case, which is the slowest.

The performance difference between a Python list comprehension and a truly vectorized operation (which `numpy` provides) becomes dramatically apparent as soon as your data size grows beyond a few thousand elements. This is why a common mantra among experienced data scientists is: "Avoid explicit Python loops for element-wise numerical operations whenever a vectorized alternative exists."

In essence, there's a fundamental disconnect: Python lists are designed as *flexible containers for arbitrary objects*, whereas statistics and machine learning fundamentally require *efficient, homogeneous data structures like vectors and matrices*. Bridging this performance and conceptual gap is the primary motivation behind the entire scientific Python ecosystem, with `numpy` at its core.

### Module 2.2: The Vector and The `ndarray`

This module introduces the first and most fundamental mathematical principle that `numpy` addresses: the concept of a vector.

**The Principle (Math): The Vector**

In mathematics, a **vector** is far more than just an ordered list of numbers. It is a precisely defined object, an element within a mathematical construct called a *vector space*. This space comes with a set of axioms (rules) that dictate how vectors behave and interact. For instance, if we consider a vector $v$ in a 3-dimensional real-numbered space ($\mathbb{R}^3$), we typically write it as:

$v = [v_1, v_2, v_3]$

Crucially, this mathematical vector inherently supports specific operations that are fundamental to numerical analysis, and which a standard Python list *does not* handle intuitively:

1.  **Scalar Multiplication:** This involves multiplying the entire vector by a single number, known as a *scalar* ($c$). The operation applies to each component of the vector:
    $c \times v = [c \cdot v_1, c \cdot v_2, c \cdot v_3]$
    *Example:* If we want to double our `revenues` vector: $2 \times [100, 200, 300] = [200, 400, 600]$

2.  **Vector Addition:** This operation combines two vectors of the *same size* (same dimensionality) by adding their corresponding components:
    $v + w = [v_1 + w_1, v_2 + w_2, v_3 + w_3]$
    *Example:* If we have `revenues = [100, 200, 300]` and `tax_adjustments = [1, 2, 3]`, their sum would be: $[100, 200, 300] + [1, 2, 3] = [101, 202, 303]$

A critical property that mathematicians rely on is *closure*. This means that when you apply these defined operations (like scalar multiplication or vector addition) to vectors within a given vector space, the result is *always another valid vector within that same space*. This closure is what makes algebraic manipulation consistent and reliable.

**The "Aha!" Moment (The Code): The `numpy.ndarray`**

The `numpy.ndarray` (N-dimensional array) was specifically engineered to bridge the gap between Python's general-purpose lists and the strict requirements of mathematical vectors and matrices. It is the foundational data structure for numerical computing in Python.

When you create a `numpy` array, for example:
`import numpy as np`
`my_array = np.array([100, 200, 300])`

You are *not* creating a collection of pointers to disparate Python objects. Instead, you are instructing `numpy` to allocate a **contiguous block of fixed-type memory**. This is analogous to a C-style array, where all elements are stored sequentially in memory and are guaranteed to be of the same data type (e.g., `int64`, `float32`).

```python
import numpy as np
my_array = np.array([100, 200, 300])
print(f"NumPy array: {my_array}")
print(f"Data type: {my_array.dtype}")
print(f"Shape: {my_array.shape}")
```

Because every element in an `ndarray` is of the *same type* and they are all stored *next to each other* in memory, `numpy` can perform mathematical operations with incredible efficiency. It achieves this by:
*   **Leveraging C/Fortran Backends:** `numpy`'s core operations are implemented in highly optimized C and Fortran code, bypassing the Python interpreter's overhead.
*   **Utilizing SIMD Instructions:** The contiguous, homogeneous memory layout allows CPUs to use Single Instruction, Multiple Data (SIMD) instructions, performing operations on multiple data points in a single clock cycle.
*   **Exploiting Cache Locality:** Data stored contiguously is more likely to reside in the CPU's fast cache, leading to much quicker access times.

Consequently, `numpy` correctly implements mathematical vector operations:
*   It performs **Scalar Multiplication** element-wise, as expected:
    ```python
    result_scalar_mult = my_array * 2
    print(f"Scalar multiplication (my_array * 2): {result_scalar_mult}") # Output: [200 400 600]
    ```
*   It performs **Vector Addition** element-wise (a "vectorized" operation):
    ```python
    another_array = np.array([1, 2, 3])
    result_vector_add = my_array + another_array
    print(f"Vector addition (my_array + another_array): {result_vector_add}") # Output: [101 202 303]
    ```

These are known as **vectorized operations**. The explicit `for` loop (which is slow in Python) is effectively "pushed down" from the Python layer into the fast, pre-compiled C code that underlies `numpy`. This is the secret to `numpy`'s speed.

An `ndarray` also carries rich metadata that directly mirrors mathematical concepts:
*   `my_array.shape`: This attribute tells you the dimensions of the array (e.g., `(3,)` for a 1D array with 3 elements, `(2, 3)` for a 2x3 matrix).
*   `my_array.dtype`: Records the precise numerical type of the elements (e.g., `int64`, `float64`), ensuring homogeneity.
*   `my_array.ndim`: Indicates the number of dimensions or axes (e.g., 1 for a vector, 2 for a matrix).

Because all this information is explicit and consistent, `numpy` can implement powerful features like *broadcasting*. Broadcasting allows `numpy` to automatically "stretch" arrays of compatible shapes to perform operations, making expressions both terse and mathematically correct. For example, `temperatures_c * 9/5 + 32` (converting Celsius to Fahrenheit) works seamlessly even if `temperatures_c` is an array of many values, because `numpy` broadcasts the scalars `9/5` and `32` across all elements.

**Why this matters (Statistician-Pythonista):**
As a statistician or data scientist, you naturally *think* in terms of vectors. Many fundamental statistical calculations are inherently vector operations. Consider calculating Z-scores, $Z = \frac{x - \mu}{\sigma}$, where $x$ is a vector of observations, $\mu$ is the mean (a scalar or vector of means), and $\sigma$ is the standard deviation (a scalar or vector of standard deviations). This formula involves vector subtraction and scalar division. `NumPy` provides a tool that behaves exactly as your mathematical intuition expects. Instead of laboriously constructing explicit Python loops, you can write the formula once, and `NumPy` applies it efficiently across an entire dataset in milliseconds, translating your mathematical thought directly into high-performance code.

### Module 2.3: The Matrix and Linear Algebra

Building upon the concept of vectors, the next crucial step in understanding numerical data structures is the matrix, which forms the backbone of linear algebra and most machine learning models.

**The Principle (Math): The Matrix**

A **matrix** is essentially a collection of vectors organized into a 2-dimensional grid (or higher dimensions for tensors). We typically denote a matrix $\mathbf{X}$ with $m$ rows and $n$ columns as an element of $\mathbb{R}^{m \times n}$, signifying that it contains real numbers and has $m$ rows and $n$ columns.

In the realm of data science, the matrix is our *fundamental data shape*. There's a widely adopted convention for interpreting its dimensions:
*   $m$ **Rows = Observations** (also known as samples, instances, records, customers, patients, etc.). Each row represents a single entity or event.
*   $n$ **Columns = Features** (also known as variables, attributes, predictors, age, price, temperature, etc.). Each column represents a specific characteristic or measurement for all observations.

Visually, a matrix $\mathbf{X}$ looks like this:

$\mathbf{X} = \begin{bmatrix}
x_{1,1} & x_{1,2} & \dots & x_{1,n} \\
x_{2,1} & x_{2,2} & \dots & x_{2,n} \\
\vdots & \vdots & \ddots & \vdots \\
x_{m,1} & x_{m,2} & \dots & x_{m,n}
\end{bmatrix}
\quad
\begin{matrix}
\leftarrow \text{Observation 1} \\
\leftarrow \text{Observation 2} \\
\dots \\
\leftarrow \text{Observation m}
\end{matrix}
$
$
\qquad \qquad \qquad
\begin{matrix}
\uparrow & \uparrow & \dots & \uparrow \\
\text{Feat. 1} & \text{Feat. 2} & \dots & \text{Feat. n}
\end{matrix}
$


Matrices provide a powerful abstraction, allowing us to represent entire datasets with a single symbol. The rich algebra of matrices then gives us a sophisticated toolkit to manipulate these datasets in principled ways: to rotate, scale, project, or combine them. Concepts like *rank*, *determinants*, and *eigenvalues*, which might initially sound abstract, tie directly to practical data science questions. For example:
*   **Rank:** Can tell you if there's redundancy in your features (e.g., if one feature is a perfect linear combination of others). A low rank might indicate multicollinearity.
*   **Determinants:** For square matrices, the determinant can indicate if a system of linear equations has a unique solution, which is crucial in many statistical models.
*   **Eigenvalues/Eigenvectors:** Form the basis of dimensionality reduction techniques like Principal Component Analysis (PCA), helping us understand the primary directions of variance in our data and how a model responds to perturbations.

**The "Why" of Machine Learning: The Dot Product**

Why do we place such immense importance on matrices and the field of linear algebra? Because linear algebra provides the mathematical language and tools for nearly all machine learning algorithms. The single most important operation in linear algebra, and thus in machine learning, is the **dot product**.

The dot product of two vectors $v$ and $w$ (which must be of the same size, $n$) is calculated by multiplying their corresponding elements and then summing these products:

$v \cdot w = \sum_{i=1}^{n} v_i \cdot w_i = v_1 w_1 + v_2 w_2 + \dots + v_n w_n$

*   **Step-by-Step Example:**
    *   Let $v = [1, 2, 3]$
    *   Let $w = [4, 5, 6]$
    *   The dot product $v \cdot w = (1 \times 4) + (2 \times 5) + (3 \times 6) = 4 + 10 + 18 = 32$

The dot product has a profound intuitive geometric interpretation: it measures how much two vectors "point in the same direction" or how much one vector contributes to the direction of another. This fundamental idea underpins many advanced concepts:
*   **Cosine Similarity:** In natural language processing or recommendation systems, the cosine of the angle between two vectors (derived from their dot product) quantifies their similarity.
*   **Projections:** In dimensionality reduction, projecting data onto a new axis involves dot products.
*   **Work in Physics:** The work done by a force is the dot product of the force and displacement vectors.

**Why this matters (Statistician-Pythonista):**
The dot product is not just an abstract mathematical operation; it is the very *engine* that drives most machine learning algorithms.

*   **A Weighted Sum:** Any time you see a weighted sum (e.g., combining features with coefficients), you are implicitly performing a dot product.
*   **Linear Regression ($y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots + \beta_n x_n$):** This ubiquitous model is elegantly expressed as a dot product. If you represent your coefficients as a vector $\beta = [\beta_0, \beta_1, \beta_2, \dots, \beta_n]$ and your features (including a constant '1' for the intercept) as $x = [1, x_1, x_2, \dots, x_n]$, then the prediction $y$ is simply $\beta \cdot x$.
*   **Neural Networks:** The core computation within each neuron and between layers of a neural network is a series of matrix multiplications. A matrix multiplication is, at its heart, a collection of many dot products performed in parallel. Each connection between neurons represents a weight, and the output is a weighted sum (dot product) of inputs.

`NumPy` provides highly optimized ways to compute the dot product, either with `np.dot(v, w)` or, more idiomatically for Python 3.5+, using the `@` operator for matrix multiplication: `v @ w`.

Scaling up from individual vector dot products to matrix multiplication gives you powerful expressions like `X @ beta`, where `X` is your design matrix (each row an observation, each column a feature) and `beta` is a vector of model parameters. The result is a vector of predictions, and the entire computation is executed with incredible speed because `numpy` delegates these operations to highly optimized Basic Linear Algebra Subprograms (BLAS) libraries, often written in low-level languages like Fortran or C. The "golden rule" for performance in numerical Python is: if you can phrase your problem as matrix algebra, you automatically inherit decades of numerical optimization for free.

### Module 2.4: The Table and The `DataFrame`

We've established `numpy` as a powerful tool for numerical computation, providing the speed and mathematical correctness that Python lists lack. However, `numpy` arrays, by themselves, have a significant limitation when dealing with real-world, heterogeneous datasets.

**The Problem with `numpy`:**

Consider a `numpy` array representing some tabular data:
`my_array = np.array([[25, 180, 75], [42, 165, 60]])`

If you access an element, say `my_array[0, 2]`, which gives `75`, what does that `75` actually represent? Is it `weight_kg`? `age`? `iq`? The array is *anonymous*. It's just a grid of numbers. It fundamentally lacks *metadata* – the descriptive labels that give meaning to our data. In a complex dataset, remembering what each column or row index signifies becomes impossible and error-prone.

**The Principle (Math): The Relational Model**

To address this, we turn to the relational model, the theoretical foundation behind all SQL databases and, by extension, most structured data handling. In this model, a "table" (or **relation**) is not merely a grid of numbers. It is a structured collection characterized by:
*   **Attributes (Columns):** Each column has a distinct *name* and represents a specific characteristic or variable (e.g., 'Age', 'Height', 'Weight'). These names provide crucial context.
*   **Tuples (Rows):** Each row represents a single, unique record or observation.
*   **Index:** A mechanism (often one or more columns) that uniquely identifies each tuple.

The genius of this model lies in its ability to give *names* and *meaning* to our data. It allows us to refer to data by its semantic label rather than its arbitrary numerical position.

**The "Aha!" Moment (The Code): The `pandas.DataFrame`**

The `pandas.DataFrame` is arguably the ultimate "Statistician-Pythonista" tool because it brilliantly *fuses* the high-performance numerical capabilities of `numpy` with the human-readable, labeled structure of the relational model. A `DataFrame` can be conceptualized as:

1.  **A NumPy Array for the Data Itself:** At its core, the `DataFrame` stores its actual data values in one or more `numpy.ndarray` objects. This means it inherits `numpy`'s speed, C-based optimizations, and vectorized operations for numerical computations.
2.  **A Dictionary-like Index for the Rows:** Each row is associated with a label (an index). This allows for fast $O(1)$ lookups by label, similar to a dictionary. For example, `df.loc['customer_id_123']` retrieves a specific customer's data efficiently.
3.  **A Dictionary-like Columns Object for the Features:** Each column has a name, allowing for fast $O(1)$ lookups by name. For instance, `df['age']` directly accesses the 'age' column.

```python
import pandas as pd
data = {'Age': [25, 42], 'Height_cm': [180, 165], 'Weight_kg': [75, 60]}
df = pd.DataFrame(data)
print(df)
print(f"\nAccessing 'Weight_kg' for the first entry: {df.loc[0, 'Weight_kg']}")
print(f"Mean age: {df['Age'].mean()}")
```

On top of this robust foundation, `pandas` layers an incredibly intuitive and powerful Application Programming Interface (API):
*   **`Series`:** A single column of a `DataFrame` is a `pandas.Series`, which can be thought of as a labeled `numpy` array. It retains the numerical efficiency while adding a meaningful name.
*   **Automatic Index Alignment:** When performing operations between `DataFrame`s or `Series`, `pandas` automatically aligns them based on their indexes, preventing common errors and simplifying complex data merges.
*   **Missing Value Handling:** `pandas` provides robust and predictable mechanisms for handling missing data (represented as `NaN`), which is ubiquitous in real-world datasets.

The `DataFrame` thus behaves like a sophisticated spreadsheet, a SQL table, and a `NumPy` array all at once.

This fusion represents a significant breakthrough for data scientists. You gain the raw computational speed and mathematical correctness of `NumPy` *plus* the human-readable, explicit labels and structured organization of a relational database.

Consider the previous anonymous `numpy` example versus a `pandas` equivalent:
*   **Less Clear (NumPy):** `my_array[:, 2].mean()` (Requires remembering that column index 2 is 'Weight_kg')
*   **Clear and Self-Documenting (Pandas):** `df['Weight_kg'].mean()`

Both operations are executed with comparable underlying efficiency because `pandas` leverages `numpy` for the heavy lifting. However, the `pandas` version is inherently more readable, maintainable, and less prone to errors because it's self-documenting.

When you chain together `pandas` methods like `df.query("age >= 18")` (filtering data), `df.assign(bmi=lambda d: d.weight_kg / (d.height_m ** 2))` (creating new features), and `df.value_counts("country")` (summarizing categorical data), you are effectively telling a coherent story about your data. The `DataFrame`'s index ensures that rows remain aligned throughout these transformations, column names preserve the meaning of your features, and the underlying `NumPy` arrays guarantee that the computations are performed efficiently.

### Module 2.5: Operations on Tables

With the `pandas.DataFrame` providing a powerful, labeled tabular data structure, we can now perform two more incredibly powerful and common statistical operations that are essential for data analysis and preparation.

**The Principle (Math): Set Theory & Joins**

Real-world data rarely resides in a single, monolithic table. Instead, information is often distributed across multiple tables (e.g., `customer_details`, `order_history`, `product_catalog`). The process of combining these disparate pieces of information relies heavily on concepts from set theory, specifically through operations known as **joins**. Joins allow us to merge tables based on common columns (keys).

Let's imagine you have `orders` in one table and `customers` in another. How do you combine them to see which customer placed which order? By using set theory on their common `customer_id` column.

*   **Inner Join ($\cap$, Intersection):** This is the most common type of join. It keeps only those rows where there is a matching key (e.g., `customer_id`) in *both* tables. Rows that do not have a match in the other table are discarded. This is useful when you only care about entities that exist in all linked datasets.
    ```python
    # Example: Find orders that have a matching customer
    # pd.merge(orders_df, customers_df, on='customer_id', how='inner')
    ```
*   **Outer Join ($\cup$, Union):** An outer join (specifically, a `full outer join`) keeps *all* rows from *both* tables. Where a match exists, the data is combined. Where a match does not exist in one of the tables, the corresponding columns are filled with `NaN` (Not a Number) or `None`. This is useful when you want to retain all information, even if it doesn't have a counterpart in the other table.
*   **Left Join:** This join keeps *all* rows from the "left" table (the first `DataFrame` specified in the merge function) and brings in matching data from the "right" table. If a row in the left table has no match in the right table, the columns from the right table will be filled with `NaN`. This is ideal when you want to enrich your primary dataset with additional information without losing any of your original records.
    ```python
    # Example: Add customer details to all orders, keeping all orders
    # pd.merge(orders_df, customers_df, on='customer_id', how='left')
    ```
*   **Right Join:** The inverse of a left join; it keeps all rows from the "right" table and brings in matching data from the "left" table.

Before performing a join, experienced data analysts meticulously check the key columns: Are there duplicate keys? Do they share the exact same data type and formatting? `pandas` allows you to enforce these data integrity checks with arguments like `validate='one_to_one'`, `validate='one_to_many'`, etc., which can save countless hours of debugging by catching mismatches early.

The `pandas.merge(df_a, df_b, on='key_column', how='inner')` function is a direct, robust, and highly optimized implementation of these mathematical set-theoretic concepts, making complex data integration straightforward.

**The Principle (Math): Split-Apply-Combine**

This strategy, popularized by Hadley Wickham, is perhaps the single most powerful and frequently used analytical pattern in data science. It provides a systematic way to break down complex analytical tasks into manageable steps:

1.  **Split:** You begin by dividing a large dataset (a `DataFrame`) into smaller, independent groups based on the unique values of one or more categorical variables. For example, you might split all customer transactions by `country` or `product_category`.
2.  **Apply:** To each of these isolated groups, you then apply a specific function or computation independently. This could be calculating the `mean(order_value)` for *each* country's group, finding the `max(sales)` for each product category, or fitting a mini-model to each subgroup.
3.  **Combine:** Finally, you reassemble the results from each group into a new, summarized `DataFrame` or `Series`. This new structure provides insights into the aggregated behavior of your data across different categories.

This "Split-Apply-Combine" strategy is *exactly* what `pandas.groupby()` was built to do, and it's one of `pandas`' most celebrated features.

```python
# Example: Calculate the average order value for each country
# df.groupby('country')['order_value'].mean()
```

This single line of code is a beautiful, Pythonic expression of a formal and incredibly powerful statistical strategy. The `GroupBy` object returned by `df.groupby()` is highly versatile and supports a rich array of subsequent operations:
*   **`agg()` (Aggregate):** Allows you to compute multiple statistics (e.g., mean, median, min, max, count) on one or more columns for each group simultaneously.
*   **`transform()`:** Returns a `Series` or `DataFrame` with the same index as the original, where each group's value is replaced by the result of the aggregation. This is invaluable for feature engineering, such as normalizing data within groups.
*   **`apply()`:** Provides the ultimate flexibility, allowing you to apply custom, user-defined functions to each group. This is used when built-in aggregation functions are not sufficient for your specific analytical needs.

### Summary of Moment 1

* **Python `list`:** A "bag of pointers". Bad for math (concatenation, not multiplication).
* **`numpy.ndarray`:** A "vector/matrix". A contiguous block of C-memory. Good for fast, vectorized *mathematical* operations (linear algebra).
* **`pandas.DataFrame`:** A "labeled matrix". The ultimate tool. It combines the *speed* of `numpy` with the *labeled* index/columns of a relational database and the *power* of `groupby` (Split-Apply-Combine).


### Module 2.6: NumPy in Practice (The "Vector" & "Matrix" Tool)

`NumPy` (Numerical Python) is the foundation of the entire scientific Python ecosystem. `pandas` is built on top of it. Mastering `NumPy` is non-negotiable.

#### 1. Creating Arrays

This is the first step. You are no longer creating a `list`; you are creating a contiguous block of memory, a `numpy.ndarray`.

  * **From a Python list:** This is the most common way. NumPy inspects your list and chooses the most appropriate, efficient data type (e.g., `int64` or `float64`).

In [ ]:
# A 1D array (a vector)
vector = np.array([10, 20, 30, 40, 50])
print(f"Vector: {vector}")
print(f"Data type: {vector.dtype}")  # Notice it's int64, a C-style fixed-size integer

# A 2D array (a matrix)
matrix = np.array([[1, 2, 3], [4, 5, 6]])
print(f"\nMatrix:\n{matrix}")
print(f"Shape: {matrix.shape}")  # (2, 6) -> 2 rows, 3 columns

Vector: [10 20 30 40 50]
Data type: int64

Matrix:
[[1 2 3]
 [4 5 6]]
Shape: (2, 3)



  * **Creating placeholders:** Often, you know the *shape* of the data you need before you know the *values*.

In [ ]:
# A 3x4 matrix of all zeros
zeros_matrix = np.zeros((3, 4))

# A 2x5 matrix of all ones
ones_matrix = np.ones((2, 5))

# Like Python's range(), but creates a NumPy array
# Creates [0, 1, 2, 3, 4]
range_array = np.arange(5)

# Statistically useful: creates N evenly spaced points
# Creates 10 points between 0 and 1 (inclusive)
linear_points = np.linspace(0, 1, 10)
print(f"\nLinspace:\n{linear_points}")


Linspace:
[0.         0.11111111 0.22222222 0.33333333 0.44444444 0.55555556
 0.66666667 0.77777778 0.88888889 1.        ]


#### 2. Vectorization in Action

Let's prove the core concept from Moment 1. We want to add 5 to every number in a large list.

In [ ]:
import time

# Create a large array with 10 million numbers
large_array = np.arange(10_000_000)
# Create an equivalent Python list
large_list = list(range(10_000_000))

# The "Bad" Way (Slow Python 'for' loop)
start_time = time.time()
py_results = []
for item in large_list:
	py_results.append(item + 5)
end_time = time.time()

print(f"Time taken for the Python loop: {end_time - start_time:.2f} seconds")

Time taken for the Python loop: 1.43 seconds


In [ ]:
# The "Good" Way (Fast NumPy Vectorization)
start_time = time.time()
np_results = large_array + 5
end_time = time.time()

print(f"Time taken for the NumPy operation: {end_time - start_time:.2f} seconds")

Time taken for the NumPy operation: 0.04 seconds


When you run this, the Python `for` loop will take a noticeable amount of time (seconds, perhaps). The NumPy operation will be *instantaneous*.

This is **vectorization**. The `large_array + 5` operation does not create a Python loop. It dispatches a single command to NumPy's underlying, pre-compiled C code, which iterates at machine speed. This is the $O(n)$ operation with a *tiny* constant $c$ that we discussed.

#### 3. Indexing and Slicing (The "Statistician's" Way)

This is where NumPy's power for data analysis truly begins.

  * **Standard Slicing:** It works like Python lists, but you can do it in multiple dimensions.

In [ ]:
matrix = np.array([
	[1, 2, 3],  # row 0
	[4, 5, 6],  # row 1
	[7, 8, 9],  # row 2
])

# Get a single element: row 1, column 2
print(f"Element [1, 2]: {matrix[1, 2]}")

Element [1, 2]: 6


In [ ]:
# Get a full row: row 0
print(f"Row 0: {matrix[0]}")  # or matrix[0, :]

Row 0: [1 2 3]


In [ ]:
# Get a full column: column 1
print(f"Column 1: {matrix[:, 1]}")

Column 1: [2 5 8]


  * **Boolean Masking (Your Most Important EDA Tool):**
    This is the core of filtering data. You don't write a `for` loop with an `if` statement. You create a "mask" and apply it to the array.

In [ ]:
data = np.array([1, 5, 2, 10, 8, 3])

# Step 1: The Condition. This is a vectorized operation.
# It returns a *new array* of booleans.
mask = data > 4
print(f"\nBoolean Mask: {mask}")

# Step 2: Apply the mask.
# This says "give me only the elements of 'data' where the 'mask' is True"
filtered_data = data[mask]
print(f"Filtered Data: {filtered_data}")


Boolean Mask: [False  True False  True  True False]
Filtered Data: [ 5 10  8]



    This is incredibly fast and readable. `data[data > 4]` is the Pythonic, statistical way to say "select all data points greater than 4".

#### 4. Broadcasting (The "Magic" and The "Danger")

**Broadcasting** is the set of rules NumPy uses to perform operations on arrays of different, but *compatible*, shapes. This is what allows `large_array + 5` to work (it "stretches" the 5) and it's what powers most feature engineering.

  * **The Concept:** When you try to add a 2D matrix (shape `(3, 3)`) and a 1D vector (shape `(3,)`), NumPy "broadcasts" (stretches or tiles) the smaller array to match the shape of the larger one.

  * **The Statistical Application (Centering Data):**
    This is the first step of `StandardScaler`. You have a matrix $X$ and you want to subtract the mean of *each feature* (column) from that feature.

In [ ]:
# A 3x2 matrix (3 observations, 2 features)
X = np.array([
	[10, 1],  # Obs 1
	[20, 2],  # Obs 2
	[30, 3],  # Obs 3
])

# Calculate the mean of each column (axis=0)
feature_means = X.mean(axis=0)
print(f"\nFeature Means: {feature_means}")
print(f"Shape of X: {X.shape}")
print(f"Shape of means: {feature_means.shape}")

X_centered = X - feature_means
print(f"Centered Data:\n{X_centered}")


Feature Means: [20.  2.]
Shape of X: (3, 2)
Shape of means: (2,)
Centered Data:
[[-10.  -1.]
 [  0.   0.]
 [ 10.   1.]]


    You just performed a complex statistical operation with one simple, readable line of code, thanks to broadcasting.

### Module 2.7: Pandas in Practice (The "Labeled Matrix" Tool)

Now we move to `pandas`. Remember, a `DataFrame` is just a `NumPy` array with two dictionaries on top: one for the **columns** (feature names) and one for the **index** (observation names).

#### 1. Creating Data Structures

  * **`pd.Series` (The 1D "Labeled Vector"):**

In [ ]:
# A Series with a custom index
s = pd.Series(
	[10.5, 9.8, 11.2],  # The data (a NumPy-like array)
	index=["Item_A", "Item_B", "Item_C"],  # The labels
)
print(f"Pandas Series:\n{s}")

# Access by label (like a dict)
print(f"\nPrice of Item_B: {s['Item_B']}")

Pandas Series:
Item_A    10.5
Item_B     9.8
Item_C    11.2
dtype: float64

Price of Item_B: 9.8


  * **`pd.DataFrame` (The 2D "Labeled Matrix"):**
    The standard way to create one is from a dictionary of columns.

In [ ]:
data_dict = {
	"age": [25, 42, 31, 55],
	"country": ["USA", "Canada", "USA", "Brazil"],
	"order_value": [100.0, 75.5, 200.0, 50.2],
}

df = pd.DataFrame(data_dict, index=["Cust_1", "Cust_2", "Cust_3", "Cust_4"])
print(f"\nDataFrame:\n{df}")


DataFrame:
        age country  order_value
Cust_1   25     USA        100.0
Cust_2   42  Canada         75.5
Cust_3   31     USA        200.0
Cust_4   55  Brazil         50.2


    This object is now a rich "spreadsheet" in your code.

#### 2. Powerful Indexing (`.loc` vs. `.iloc`)

This is the most common point of confusion for new users. Mastering it is essential.

  * **`.loc` is for LABELS.** (Think "location")
    It uses the *names* in the index and columns.

In [ ]:
# Get a single row by its *index label*
# Output is a Series
cust_2_data = df.loc["Cust_2"]
print(f"\nCustomer 2 Data:\n{cust_2_data}")

# Get a single column by its *column label*
# Output is a Series
ages = df.loc[:, "age"]  # : means "all rows"
print(f"\nAges:\n{ages}")


Customer 2 Data:
age                42
country        Canada
order_value      75.5
Name: Cust_2, dtype: object

Ages:
Cust_1    25
Cust_2    42
Cust_3    31
Cust_4    55
Name: age, dtype: int64


  * **`.iloc` is for INTEGERS.** (Think "integer-location")
    It uses the Python *position* (0, 1, 2...). It behaves just like NumPy array slicing.

In [ ]:
# Get the *first row* (position 0)
first_cust_data = df.iloc[0]

# Get the *second column* (position 1)
countries = df.iloc[:, 1]


  * **The "Statistician-Pythonista" Way (Boolean Masking with `.loc`):**
    This is how you combine `pandas` labeling with `NumPy` boolean masking.

    **Question:** "Show me the `age` and `order_value` for all customers in the 'USA'."

In [ ]:
# Step 1: Create the boolean mask
mask = df["country"] == "USA"
print(f"\nMask for USA:\n{mask}")

# Step 2: Apply the mask with .loc
# .loc[<row_mask>, <column_labels>]
result = df.loc[mask, ["age", "order_value"]]
print(f"\nUSA Customers:\n{result}")


Mask for USA:
Cust_1     True
Cust_2    False
Cust_3     True
Cust_4    False
Name: country, dtype: bool

USA Customers:
        age  order_value
Cust_1   25        100.0
Cust_3   31        200.0


    This is one of the most powerful and readable patterns in all of data analysis.


#### 3. Split-Apply-Combine in Action (`groupby`)

This is the programmatic implementation of the statistical strategy we discussed.

**Question:** "What is the *average order value* and *total number of customers* for each `country`?"

In [ ]:
# Step 1: SPLIT the DataFrame into groups
grouped = df.groupby("country")

# 'grouped' is a special object. You can't see it, but it contains
# two "sub-DataFrames", one for 'USA' and one for 'Canada' / 'Brazil'.

# Step 2 & 3: APPLY a function and COMBINE the results
# Calculate the mean for all numeric columns
print(f"\nMean values by country:\n{grouped.mean(numeric_only=True)}")

# More powerful: Use .agg() for custom aggregations
summary = grouped.agg(
	total_revenue=("order_value", "sum"),
	average_revenue=("order_value", "mean"),
	customer_count=("age", "count"),  # 'age' is arbitrary, just need to count rows
)
print(f"\nCountry Summary:\n{summary}")


Mean values by country:
          age  order_value
country                   
Brazil   55.0         50.2
Canada   42.0         75.5
USA      28.0        150.0

Country Summary:
         total_revenue  average_revenue  customer_count
country                                                
Brazil            50.2             50.2               1
Canada            75.5             75.5               1
USA              300.0            150.0               2


#### 4\. Reshaping Data (Joins & Pivots)

  * **`pd.merge()` (The Relational Join):**
    Let's use the two tables from Moment 1.


In [ ]:
customers_df = pd.DataFrame({
	"customer_id": ["c1", "c2", "c3", "c4"],
	"name": ["Alice", "Bob", "Charlie", "David"],
})

orders_df = pd.DataFrame({
	"order_id": [1001, 1002, 1003],
	"customer_id": ["c2", "c3", "c2"],
	"amount": [75.5, 200.0, 50.0],
})

# Perform an INNER join
# Keep only rows where 'customer_id' exists in *both* tables
joined_df = pd.merge(
	customers_df,  # Left table
	orders_df,  # Right table
	on="customer_id",  # The key to match
	how="inner",  # The type of join
)
print(f"\nJoined DataFrame (Inner):\n{joined_df}")


Joined DataFrame (Inner):
  customer_id     name  order_id  amount
0          c2      Bob      1001    75.5
1          c2      Bob      1003    50.0
2          c3  Charlie      1002   200.0


    This is a direct, robust implementation of relational algebra.

  * **`pd.pivot_table()` (The Ultimate Summary Tool):**
    This takes a "long" format table and makes it "wide". It is like `groupby` but it creates a 2D matrix.

    **Question:** "Show me the *total* sales for each `country` (rows) broken down by `product_category` (columns)."

In [ ]:
# A slightly more complex dataset
sales_df = pd.DataFrame({
	"country": ["USA", "USA", "Canada", "USA", "Canada", "Brazil"],
	"product": ["A", "B", "A", "B", "B", "A"],
	"sales": [100, 150, 80, 200, 50, 60],
})

pivot = pd.pivot_table(
	sales_df,
	values="sales",  # The numbers to aggregate
	index="country",  # The rows
	columns="product",  # The columns
	aggfunc="sum",  # The function (Apply step)
	fill_value=0,  # Fill in missing (e.g., Brazil/Product B) with 0
)
print(f"\nSales Pivot Table:\n{pivot}")


Sales Pivot Table:
product    A    B
country          
Brazil    60    0
Canada    80   50
USA      100  350


### Summary of Moment 2

  * Create and manipulate `numpy` arrays for fast, mathematical operations.
  * Use **vectorization** (`arr + 5`) instead of slow Python loops.
  * Use **boolean masking** (`arr[arr > 4]`) to filter data.
  * Use **broadcasting** (`X - X.mean(axis=0)`) to perform feature engineering.
  * Create and manipulate `pandas` DataFrames, the labeled matrix.
  * Use `.loc` (labels) and `.iloc` (integers) to select data.
  * Use `.groupby()` to implement the **Split-Apply-Combine** strategy.
  * Use `.merge()` and `pivot_table()` to join and reshape data.


### Module 2.9: Application — The E-Commerce Analysis Pipeline

**The Dataset: The Olist E-commerce Dataset**

  * **Source:** Publicly available on Kaggle. (It is a large, famous dataset of Brazilian e-commerce orders).
  * **Why it's perfect:** It is not one file. It is a relational database spread across multiple CSV files. This *forces* you to think about data relationships and build an efficient pipeline.
  * **Your Task:** I expect you to download the CSVs from Kaggle and place them in a local directory (e.g., `data/`).

**The "Minimal Starter" (Our Database Schema):**

These are the key files and their "join keys":

  * `olist_customers_dataset.csv`
      * `customer_id` (Unique ID for the customer)
      * `customer_unique_id` (The *actual* unique ID for a person, as one person can have multiple `customer_id`s for different orders)
      * `customer_zip_code_prefix`, `customer_city`, `customer_state`
  * `olist_orders_dataset.csv`
      * `order_id` (Unique ID for an order)
      * `customer_id` (Links to the customers table)
      * `order_status` ('delivered', 'shipped', etc.)
      * `order_purchase_timestamp` (and other timestamps)
  * `olist_order_items_dataset.csv` (The "Junction Table")
      * `order_id` (Links to the orders table)
      * `product_id` (Links to the products table)
      * `price` (Price of the item)
      * `freight_value` (Shipping cost for this item)
  * `olist_products_dataset.csv`
      * `product_id` (Unique ID for a product)
      * `product_category_name`
      * `product_weight_g`, `product_length_cm`, etc.

Your code will be complex, and it will be tempting to do it all in one cell. *Do not*. Your goal is to build functions.

### Tasks

**Task 1: The "DRY" Data Loader**

  * **Objective:** Do not write `pd.read_csv()` 8 times in 8 different cells. Create a single, reusable function to load your data.
  * **Implementation:**
    1.  Create a dictionary (a "file map") that maps a friendly name to its filename.
        ```python
        FILE_MAP = {
            'customers': 'olist_customers_dataset.csv',
            'orders': 'olist_orders_dataset.csv',
            'order_items': 'olist_order_items_dataset.csv',
            'products': 'olist_products_dataset.csv'
            # Add other files as needed
        }
        DATA_PATH = 'data/' # Or your path
        ```
    2.  Write a function `load_data(file_map, data_path)` that:
          * Takes the `file_map` and `data_path` as arguments.
          * Loops through the `file_map`.
          * Loads each CSV into a pandas DataFrame.
          * Stores them in a *new* dictionary where the key is the friendly name and the value is the DataFrame (e.g., `{'customers': df_customers, 'orders': df_orders, ...}`).
          * Returns this dictionary of DataFrames.
  * **Deliverable:** The definition of your `load_data` function and the code you use to call it.

In [ ]:
from pathlib import Path


def load_data(data_path: str) -> dict:
	"""
	Load CSV files from the specified directory into a dictionary of DataFrames.

	Scans the directory for CSV files, cleans their names to create friendly keys,
	and loads them into pandas DataFrames.

	Parameters
	----------
	data_path : str
		The path to the directory containing the CSV files.

	Returns
	-------
	dict
		A dictionary where keys are cleaned file names (e.g., 'customers') and values are pandas DataFrames.

	Raises
	------
	ValueError
		If the path does not exist or is not a directory.
	"""
	path = Path(data_path)

	# Check if the path exists and is a directory
	if not path.exists() or not path.is_dir():
		raise ValueError(
			f"The path '{data_path}' does not exist or is not a directory."
		)

	# Build file_map: cleaned keys to original filenames
	file_map = {
		file.name.replace("olist_", "")
		.replace("_dataset.csv", "")
		.replace(".csv", ""): file.name
		for file in path.iterdir()
		if file.is_file() and file.suffix == ".csv"  # Extra filter for only CSVs
	}

	# Build data_map: cleaned keys to loaded DataFrames
	data_map = {key: pd.read_csv(path / filename) for key, filename in file_map.items()}

	return data_map

In [ ]:
from urllib.parse import urlparse

import requests


def load_data_github(github_folder_url: str) -> dict:
	"""
	Load CSV files from a GitHub folder URL into a dictionary of DataFrames.

	Parses the GitHub URL, fetches the folder contents via the GitHub API,
	and loads CSV files directly from their raw download URLs.

	Parameters
	----------
	github_folder_url : str
	    The GitHub URL pointing to a folder containing CSV files
	    (e.g., https://github.com/user/repo/tree/branch/path/to/folder).

	Returns
	-------
	dict
	    A dictionary where keys are cleaned file names (e.g., 'customers')
	    and values are pandas DataFrames.

	Raises
	------
	ValueError
	    If the URL is invalid, the folder does not exist, or no CSV files are found.
	"""
	# Parse the GitHub URL to extract owner, repo, branch, and folder path
	parsed = urlparse(github_folder_url)
	if parsed.hostname != "github.com":
		raise ValueError("URL must be a valid GitHub repository URL.")

	path_parts = parsed.path.strip("/").split("/")
	if len(path_parts) < 4 or path_parts[2] != "tree":
		raise ValueError(
			"URL must point to a GitHub folder (e.g., /tree/branch/path/to/folder)."
		)
	owner = path_parts[0]
	repo = path_parts[1]
	branch = path_parts[3]  # Fixed: Branch is at index 3
	folder_path = "/".join(path_parts[4:])  # Fixed: Folder starts at index 4

	# Construct GitHub API URL to list folder contents
	api_url = f"https://api.github.com/repos/{owner}/{repo}/contents/{folder_path}?ref={branch}"

	# Fetch the contents
	response = requests.get(api_url)
	if response.status_code != 200:
		raise ValueError(
			f"Failed to fetch GitHub contents: {response.status_code} - {response.text}"
		)
	contents = response.json()
	if not isinstance(contents, list):
		raise ValueError(
			"GitHub API did not return a list of files (folder may not exist)."
		)

	# Build file_map: cleaned keys to raw download URLs
	file_map = {}
	for item in contents:
		if item["type"] == "file" and item["name"].endswith(".csv"):
			cleaned_key = (
				item["name"]
				.replace("olist_", "")
				.replace("_dataset.csv", "")
				.replace(".csv", "")
			)
			raw_url = item["download_url"]
			file_map[cleaned_key] = raw_url
	if not file_map:
		raise ValueError("No CSV files found in the GitHub folder.")

	data_map = {key: pd.read_csv(url) for key, url in file_map.items()}

	return data_map

In [ ]:
github_url = "https://github.com/jhlopesalves/data-science-notebooks/tree/main/Python/statistics/exploratory_data_analysis/eda_modules/data/olist"

data_map = load_data_github(github_url)

# data_map is a dictionary, so print its keys
print("Loaded data keys:", list(data_map.keys()))

Loaded data keys: ['customers', 'geolocation', 'order_items', 'order_payments', 'order_reviews', 'orders', 'products', 'sellers', 'product_category_name_translation']


**Task 2: The "Relational" Master Table**

  * **Objective:** Combine the separate tables into one "master" analysis table that contains all relevant information for "delivered" orders.
  * **Implementation:**
    1.  Using the dictionary of DataFrames from Task 1, write a function `create_master_table(data_dict)`.
    2.  Inside this function, perform a sequence of `pd.merge()` calls. The logical flow is:
        `orders` $\rightarrow$ `order_items` (on `order_id`)
        $\rightarrow$ `products` (on `product_id`)
        $\rightarrow$ `customers` (on `customer_id`)
    3.  **Crucial Filter:** Before or after merging, filter the data for `order_status == 'delivered'`. An analysis of cancelled or unshipped orders is different.
  * **Deliverable:** The definition of your `create_master_table` function and a printout of the `.info()` and `.head()` of your final master DataFrame.

In [ ]:
from pandas import DataFrame


def create_master_table(data_dict: dict) -> DataFrame:
	"""
	Create a master table by merging multiple DataFrames from the Olist dataset.

	This function performs a series of left joins to combine order, item, product,
	customer, payment, and category translation data into a single DataFrame.
	It filters for orders with status 'delivered' to focus on completed transactions.

	Parameters
	----------
	data_dict : dict
		A dictionary containing DataFrames with keys such as 'orders', 'order_items',
		'products', 'customers', 'order_payments', and 'product_category_name_translation'.

	Returns
	-------
	pd.DataFrame
		The merged master DataFrame containing all relevant columns from the input DataFrames.
	"""
	# Required keys for the merge operation
	required_keys = [
		"orders",
		"order_items",
		"products",
		"customers",
		"order_payments",
		"product_category_name_translation",
	]

	# Check if all required DataFrames are present
	missing_keys = [key for key in required_keys if key not in data_dict]
	if missing_keys:
		raise KeyError(f"Missing required DataFrames in data_dict: {missing_keys}")

	# Check if DataFrames are not empty
	for key in required_keys:
		if data_dict[key].empty:
			raise ValueError(f"DataFrame '{key}' is empty")

	# 1. Start with the 'orders' dataframe
	merged_df = data_dict["orders"].copy()

	# Filter for orders with status 'delivered'
	merged_df = merged_df[merged_df["order_status"] == "delivered"].copy()

	# 2. Merge with 'order_items' on 'order_id'
	merged_df = pd.merge(merged_df, data_dict["order_items"], on="order_id", how="left")

	# 3. Merge with 'products' on 'product_id'
	merged_df = pd.merge(merged_df, data_dict["products"], on="product_id", how="left")

	# 4. Merge with 'customers' on 'customer_id'
	merged_df = pd.merge(
		merged_df, data_dict["customers"], on="customer_id", how="left"
	)

	# 5. Merge with 'order_payments' on 'order_id'
	# Use 'how='left'' to ensure orders without payment info are kept.
	merged_df = pd.merge(
		merged_df, data_dict["order_payments"], on="order_id", how="left"
	)

	# 6. Merge with 'product_category_name_translation' on 'product_category_name'
	# This translates category names from Portuguese to English.
	merged_df = pd.merge(
		merged_df,
		data_dict["product_category_name_translation"],
		on="product_category_name",
		how="left",
	)

	return merged_df

In [ ]:
# Call the function with the loaded data_map
master_df = create_master_table(data_map)

display(master_df.head())

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,product_width_cm,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,payment_sequential,payment_type,payment_installments,payment_value,product_category_name_english
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,87285b34884572647811a353c7ac498a,...,13.0,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,credit_card,1.0,18.12,housewares
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,87285b34884572647811a353c7ac498a,...,13.0,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,3.0,voucher,1.0,2.00,housewares
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,87285b34884572647811a353c7ac498a,...,13.0,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,2.0,voucher,1.0,18.59,housewares
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1,595fac2a385ac33a80bd5114aec74eb8,...,19.0,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.0,boleto,1.0,141.46,perfumery
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1,aa4383b373c6aca5d8797843e5594415,...,21.0,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.0,credit_card,3.0,179.12,auto


In [ ]:
display(master_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 115038 entries, 0 to 115037
Data columns (total 31 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   order_id                       115038 non-null  object 
 1   customer_id                    115038 non-null  object 
 2   order_status                   115038 non-null  object 
 3   order_purchase_timestamp       115038 non-null  object 
 4   order_approved_at              115023 non-null  object 
 5   order_delivered_carrier_date   115036 non-null  object 
 6   order_delivered_customer_date  115030 non-null  object 
 7   order_estimated_delivery_date  115038 non-null  object 
 8   order_item_id                  115038 non-null  int64  
 9   product_id                     115038 non-null  object 
 10  seller_id                      115038 non-null  object 
 11  shipping_limit_date            115038 non-null  object 
 12  price                         

None

**Task 3: The "Split-Apply-Combine" Customer Summary**

  * **Objective:** Move from an order-level view to a customer-level view. We want one row for each *unique customer* (`customer_unique_id`) with their summary statistics.
  * **Implementation:**
    1.  Write a function `summarize_customers(master_df)`.
    2.  Inside, use `groupby('customer_unique_id')`.
    3.  Use the `.agg()` method to calculate the following for each customer:
          * `total_spending`: The sum of `price`.
          * `total_freight`: The sum of `freight_value`.
          * `total_orders`: The count of unique `order_id`s.
          * `first_order_date`: The `min` of `order_purchase_timestamp`.
          * `last_order_date`: The `max` of `order_purchase_timestamp`.
  * **Deliverable:** The function definition and the `.head()` of the resulting customer summary DataFrame.

In [ ]:
def summarise_customers(master_df: DataFrame) -> DataFrame:
	"""
	Summarize customer data by aggregating key metrics for each unique customer.

	This function groups the master DataFrame by 'customer_unique_id' and computes
	total spending, total freight, total orders, first order date, and last order date
	for each customer.

	Parameters
	----------
	master_df : pd.DataFrame
		The master DataFrame containing order and customer information.

	Returns
	-------
	pd.DataFrame
		A DataFrame with 'customer_unique_id' as index and aggregated columns:
		- total_spending: Sum of 'price' for each customer.
		- total_freight: Sum of 'freight_value' for each customer.
		- total_orders: Number of unique 'order_id' for each customer.
		- first_order_date: Minimum 'order_purchase_timestamp' for each customer.
		- last_order_date: Maximum 'order_purchase_timestamp' for each customer.
	"""
	# Ensure order_purchase_timestamp is datetime for proper min/max
	master_df["order_purchase_timestamp"] = pd.to_datetime(
		master_df["order_purchase_timestamp"]
	)

	# Group by customer_unique_id and aggregate
	customer_summary = master_df.groupby("customer_unique_id").agg({
		"price": "sum",
		"freight_value": "sum",
		"order_id": "nunique",
		"order_purchase_timestamp": ["min", "max"],
	})

	# Flatten the MultiIndex columns
	customer_summary.columns = [
		"total_spending",
		"total_freight",
		"total_orders",
		"first_order_date",
		"last_order_date",
	]

	return customer_summary

In [ ]:
summary_customers = summarise_customers(master_df)
summary_customers.head()

,total_spending,total_freight,total_orders,first_order_date,last_order_date
customer_unique_id,,,,,
0000366f3b9a7992bf8c76cfdf3221e2,129.90,12.00,1,2018-05-10 10:56:27,2018-05-10 10:56:27
0000b849f77a49e4a4ce2b2a4ca5be3f,18.90,8.29,1,2018-05-07 11:11:27,2018-05-07 11:11:27
0000f46a3911fa3c0805444483337064,69.00,17.22,1,2017-03-10 21:05:03,2017-03-10 21:05:03
0000f6ccb0745a6a4b88665a16c9f078,25.99,17.63,1,2017-10-12 20:29:41,2017-10-12 20:29:41
0004aac84e0df4da2b147fca70cf8255,180.00,16.89,1,2017-11-14 19:45:42,2017-11-14 19:45:42


**Task 4: The "Vectorized" Feature Engineering**

  * **Objective:** Create new, insightful features in your customer summary table using `pandas` (which uses `numpy` vectorized operations under the hood).
  * **Implementation:**
    1.  First, ensure your date columns (`first_order_date`, `last_order_date`) are in the correct `datetime` format using `pd.to_datetime()`.
    2.  Add these new columns to your customer summary DataFrame:
          * `avg_order_value`: `total_spending` / `total_orders`. (This is a vectorized operation).
          * `customer_lifetime_days`: (`last_order_date` - `first_order_date`). This is a vectorized `datetime` subtraction.
          * Extract the *numeric* days from this by using the `.dt.days` accessor.
  * **Deliverable:** The code you used to engineer these features and a `.describe()` of your final customer summary table.

In [ ]:
# Ensure date columns are datetimes (no-op if they already are)
summary_customers["first_order_date"] = pd.to_datetime(
	summary_customers["first_order_date"]
)
summary_customers["last_order_date"] = pd.to_datetime(
	summary_customers["last_order_date"]
)

# Vectorized average order value
summary_customers["avg_order_value"] = (
	summary_customers["total_spending"] / summary_customers["total_orders"]
)

# Compute lifetime as a Timedelta then extract days
summary_customers["customer_lifetime_days"] = (
	summary_customers["last_order_date"] - summary_customers["first_order_date"]
).dt.days

In [ ]:
summary_customers.describe()

,total_spending,total_freight,total_orders,first_order_date,last_order_date,avg_order_value,customer_lifetime_days
count,93358.000000,93358.000000,93358.000000,93358,93358,93358.000000,93358.000000
mean,147.967648,24.638476,1.033420,2018-01-01 12:18:32.380503296,2018-01-04 03:47:36.045363200,143.648494,2.634032
min,0.850000,0.000000,1.000000,2016-09-15 12:16:38,2016-09-15 12:16:38,0.850000,0.000000
25%,48.900000,14.100000,1.000000,2017-09-13 15:41:22.500000,2017-09-17 18:25:15,47.900000,0.000000
50%,89.900000,17.680000,1.000000,2018-01-20 06:48:07,2018-01-23 00:12:12.500000,89.497500,0.000000
75%,159.800000,26.550000,1.000000,2018-05-05 14:13:38.249999872,2018-05-07 17:22:34.750000128,153.000000,0.000000
max,13440.000000,1794.960000,15.000000,2018-08-29 15:00:37,2018-08-29 15:00:37,13440.000000,633.000000
std,244.029704,27.054433,0.209097,NaN,NaN,237.867264,24.955822


**Task 5: The "Reshaping" Analysis (Pivot Table)**

  * **Objective:** Create a summary matrix showing total sales by product category over time.
  * **Implementation:**
    1.  Your `master_table` (from Task 2) has `order_purchase_timestamp`, `product_category_name`, and `price`.
    2.  First, create new `year` and `month` columns from the `order_purchase_timestamp` (use the `.dt.year` and `.dt.month` accessors).
    3.  Use `pd.pivot_table()` to create a matrix where:
          * `index` is `product_category_name`
          * `columns` is `year` (or a combination of year-month)
          * `values` is `price`
          * `aggfunc` is `'sum'`
          * `fill_value` is `0`
  * **Deliverable:** The `pivot_table` code and the resulting table (or its `.head()`).

In [ ]:
master_df["year"] = master_df["order_purchase_timestamp"].dt.year
master_df["month"] = master_df["order_purchase_timestamp"].dt.month

In [ ]:
pivot = pd.pivot_table(
	master_df,
	index="product_category_name",
	columns="year",
	values="price",
	aggfunc="sum",
	fill_value=0,
)
pivot.head()

year,2016,2017,2018
product_category_name,,,
agro_industria_e_comercio,0.0,29223.18,52654.40
alimentos,79.9,8821.04,20556.24
alimentos_bebidas,0.0,8188.38,7930.55
artes,0.0,9143.69,15032.94
artes_e_artesanato,0.0,151.89,1662.12


**Task 6 (Advanced Challenge): The "SOC" Refactor**

  * **Objective:** Apply the **Separation of Concerns (SOC)** principle by refactoring your entire workflow into a Python `class`.
  * **Implementation:**
      * Create a file `olist_analyzer.py`.
      * Inside, create a class named `OlistAnalyzer`.
      * The `__init__(self, data_path)` method should call a (private) `_load_data()` method (your Task 1 function).
      * Create a public method `.build_master_table(self)` (your Task 2 function).
      * Create a public method `.build_customer_summary(self)` (your Task 3 & 4 functions).
      * Create a public method `.get_sales_pivot(self)` (your Task 5 function).
      * The class should store the DataFrames as attributes (e.g., `self.data_dict`, `self.master_df`).
  * **Deliverable:** The code for your `olist_analyzer.py` file and the *new*, clean notebook code to use it:
    ```python
    from olist_analyzer import OlistAnalyzer

    analyzer = OlistAnalyzer(data_path='data/')
    customer_summary = analyzer.build_customer_summary()
    print(customer_summary.head())

    sales_pivot = analyzer.get_sales_pivot()
    print(sales_pivot.head())
    ```

### Task 6 Implementation: Using the OlistAnalyzer Class

Now we can use the refactored `OlistAnalyzer` class to perform all our analysis tasks in a clean, organized way. The class encapsulates all the data loading, merging, and analysis logic.

In [ ]:
from helpers.olist_analyzer import OlistAnalyzer

# Initialize the analyzer with GitHub data source


In [ ]:
# Build the master table

# Display basic information


In [ ]:
# Build the customer summary with engineered features


In [ ]:
# Get the sales pivot table


In [ ]:
# Bonus: Get top 10 product categories by revenue

In [ ]:
# View overall data information


### Alternative: Using Local Data

If you have the data stored locally, you can use the analyzer with a local path:

In [ ]:
# Example with local data (uncomment if you have data locally)
# local_analyzer = OlistAnalyzer(data_path='data/olist/', use_github=False)
# local_customer_summary = local_analyzer.build_customer_summary()
# print(local_customer_summary.head())